In [1]:
!pip install torch transformers datasets scikit-learn pandas numpy matplotlib

  Using cached matplotlib-3.11.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached contourpy-1.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.66.1-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (130 kB)
  Using cached kiwisolver-1.5.1-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 12.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 5.5 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 207.7 kB/s  0:0

In [1]:
import os, re, random, time
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from types import SimpleNamespace
from datasets import load_dataset
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

/common/home/users/l/lorenzof.w.2023/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# The six dialogue acts the NLU predicts
ACTS = ["propose", "counter", "accept", "reject", "inquire", "vague"]
A2I = {a: i for i, a in enumerate(ACTS)}

# All settings in one place. Change epochs here after the pilot.
cfg = SimpleNamespace(epochs=8, bs=32, lr=3e-5, max_len=192, out="nlu_pilot")

dev = "cuda" if torch.cuda.is_available() else "cpu"
print(dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

cuda NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition MIG 1g.24gb


In [4]:
raw = load_dataset("stanfordnlp/craigslist_bargains", revision="refs/convert/parquet")
d = raw["train"][0]
for t in range(min(8, len(d["utterance"]))):
    print(t, d["agent_turn"][t], d["dialogue_acts"]["intent"][t], "|", d["utterance"][t])

0 0 init-price | Hi, not sure if the charger would work for my car. Can you sell it to me for $5?
1 1 unknown | It will work, i have never seen a car without a cigarette lighter port.\
2 0 insist | Still, can I buy it for $5? I'm on a tight budge
3 1 counter-price | I think the lowest I would want to go is 8. 
4 0 counter-price | How about $6 and I pick it up myself? It'll save you shipping to me.
5 1 counter-price | 7, and we have a deal.
6 0 agree | Eh, fine. $7.
7 0 offer | 


In [5]:
# Map the dataset's original intents onto our six acts
INTENT_MAP = {
    "init-price": "propose", "counter-price": "counter", "insist": "counter",
    "agree": "accept", "accept": "accept", "disagree": "reject", "reject": "reject",
    "inquiry": "inquire", "vague-price": "vague", "unknown": "vague",
}

# Prices (digits, $ amounts, number words) are replaced by <PRICE>,
# so the model learns the structure of the exchange, not specific numbers
NUM_WORDS = r"(?:one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|fifteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred|thousand)"
PRICE_RE = re.compile(
    rf"(\$\s?\d[\d,]*(?:\.\d+)?|\b\d[\d,]*(?:\.\d+)?\s?(?:k|bucks|dollars?)?\b|\b{NUM_WORDS}(?:[\s-]{NUM_WORDS})*\b)", re.I)

# Short agreement phrases, used only to relabel agreement-like turns
AGREE_RE = re.compile(r"\b(ok(ay)?|deal|sounds good|works for me|that works|fine|sure|agreed?|you got it|done)\b", re.I)

def mask_prices(t): return PRICE_RE.sub("<PRICE>", t)

def valid_price(p):
    # the dataset's price field can be None, NaN or negative when there is no price
    return p is not None and not (isinstance(p, float) and (np.isnan(p) or p < 0))

print(mask_prices("I can do fifty five or $60, maybe 45 bucks"))

I can do <PRICE> or <PRICE> maybe <PRICE>


In [6]:
def build_examples(split, max_hist=3):
    rows = []
    for d_idx, d in enumerate(split):
        roles, acts = d["agent_info"]["Role"], d["dialogue_acts"]
        utts, turns = d["utterance"], d["agent_turn"]
        for t in range(len(utts)):
            if roles[turns[t]] != "seller": continue             # NLU reads seller messages only
            lab = INTENT_MAP.get(acts["intent"][t])
            if lab is None or not utts[t].strip(): continue      # skip turns with no negotiation content

            # up to 3 previous turns, with speaker tags and masked prices
            hist = [("BUYER" if roles[turns[h]] == "buyer" else "SELLER", mask_prices(utts[h]))
                    for h in range(max(0, t - max_hist), t)]

            # was there an offer in the previous turn?
            prev_has_price = t > 0 and (PRICE_RE.search(utts[t-1]) is not None
                                        or valid_price(acts["price"][t-1]))
            text, orig = utts[t], lab

            # relabel: "accept" with nothing to accept -> vague;
            # short agreement right after an offer -> accept
            if lab == "accept" and not prev_has_price:
                lab = "vague"
            elif lab in ("vague", "inquire") and prev_has_price and len(text.split()) <= 6 \
                    and AGREE_RE.search(text) and "?" not in text:
                lab = "accept"

            rows.append(dict(dialogue=d_idx, turn=t, text=mask_prices(text),
                             hist=hist, label=lab, orig=orig))
    return rows

In [7]:
tr_rows = build_examples(raw["train"])
# offset the ids so train and validation dialogues never share an id
va_rows = [{**r, "dialogue": r["dialogue"] + 10_000_000} for r in build_examples(raw["validation"])]
all_rows = tr_rows + va_rows

rng = np.random.RandomState(42)
dlg = np.array(sorted({r["dialogue"] for r in all_rows}))
rng.shuffle(dlg)
n = len(dlg)
test_ids = set(dlg[:int(.1 * n)])
val_ids  = set(dlg[int(.1 * n):int(.2 * n)])

# whole dialogues go to one split only, so there is no leakage
tr = [r for r in all_rows if r["dialogue"] not in test_ids | val_ids]
va = [r for r in all_rows if r["dialogue"] in val_ids]
te = [r for r in all_rows if r["dialogue"] in test_ids]
print(len(tr), len(va), len(te))
print(pd.Series([r["label"] for r in tr]).value_counts())

12222 1540 1532
vague      3940
counter    3510
propose    1399
reject     1381
accept     1061
inquire     931
Name: count, dtype: int64


In [8]:
# How much did relabelling change? (history helps partly by construction if this is large)
print(pd.crosstab(pd.Series([r["orig"] for r in tr], name="original"),
                  pd.Series([r["label"] for r in tr], name="relabelled")))
print("fraction changed:", np.mean([r["orig"] != r["label"] for r in tr]).round(3))

# How often does the k=3 input exceed max_len? (if > ~2%, raise max_len)
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tok_len(r, k=3):
    ctx = " ".join(f"[{w}] {m}" for w, m in r["hist"][-k:])
    return len(tok("[SELLER] " + r["text"], ctx)["input_ids"])
print("fraction over max_len:", (np.array([tok_len(r) for r in tr[:2000]]) > cfg.max_len).mean().round(3))

relabelled  accept  counter  inquire  propose  reject  vague
original                                                    
accept         726        0        0        0       0     91
counter          0     3510        0        0       0      0
inquire          3        0      931        0       0      0
propose          0        0        0     1399       0      0
reject           0        0        0        0    1381      0
vague          332        0        0        0       0   3849
fraction changed: 0.035
fraction over max_len: 0.008


In [10]:
class DS(Dataset):
    def __init__(self, rows, k): self.rows, self.k = rows, k
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        # k=0 gives no history; k>0 adds the last k turns as the second sentence
        ctx = " ".join(f"[{w}] {m}" for w, m in r["hist"][-self.k:]) if self.k > 0 else ""
        enc = tok("[SELLER] " + r["text"], ctx or None, truncation="longest_first",
                  max_length=cfg.max_len, padding="max_length", return_tensors="pt")
        item = {a: v.squeeze(0) for a, v in enc.items()}
        item["labels"] = torch.tensor(A2I[r["label"]])
        return item

x = DS(tr, 2)[5]
print(tok.decode(x["input_ids"], skip_special_tokens=False)[:300], "->", ACTS[x["labels"]])

[CLS] [ seller ] no, that offer is too low. the board is pretty much brand new as it ' s been ridden only < price > or < price > times. the bone bearings are brand new. [SEP] [ seller ] i do not know specifically but the brand is a gravity skateboard. [ buyer ] the wheels seem nice on it, but they c -> vague


In [11]:
@torch.no_grad()
def get_logits(model, dl):
    model.eval(); L, Y = [], []
    for b in dl:
        y = b.pop("labels")
        L.append(model(**{k: v.to(dev) for k, v in b.items()}).logits.cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

def fit_temperature(logits, y):
    # temperature scaling: one number T that divides the logits, fitted on validation only
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad(); loss = nn.functional.cross_entropy(logits / logT.exp(), y); loss.backward(); return loss
    opt.step(closure)
    return float(logT.exp())

def ece(probs, y, bins=10):
    # expected calibration error: average gap between confidence and accuracy
    conf, pred = probs.max(1); acc = (pred == y).float(); e = 0.0
    edges = torch.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.float().mean() * (acc[m].mean() - conf[m].mean()).abs()
    return float(e)

In [12]:
def run(k, seed, tr, va, te):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=len(ACTS)).to(dev)
    dl_tr = DataLoader(DS(tr, k), batch_size=cfg.bs, shuffle=True)
    dl_va, dl_te = DataLoader(DS(va, k), batch_size=64), DataLoader(DS(te, k), batch_size=64)

    # class-weighted loss, because accept/reject/inquire are rarer
    counts = np.bincount([A2I[r["label"]] for r in tr], minlength=len(ACTS)) + 1
    w = torch.tensor(counts.sum() / (len(ACTS) * counts), dtype=torch.float).to(dev)
    lossf = nn.CrossEntropyLoss(weight=w)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
    total = len(dl_tr) * cfg.epochs
    # 10% warmup, then linear decay to zero
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / (0.1 * total), 1.0) * max(0.0, 1 - s / total))

    best_f1, best_state = -1, None
    for ep in range(cfg.epochs):
        model.train()
        for b in dl_tr:
            y = b.pop("labels").to(dev)
            loss = lossf(model(**{a: v.to(dev) for a, v in b.items()}).logits, y)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        # keep the epoch with the best validation macro-F1
        lv, yv = get_logits(model, dl_va)
        f1 = f1_score(yv, lv.argmax(1), average="macro")
        print(f"k={k} seed={seed} ep={ep+1} val macro-F1={f1:.4f}")
        if f1 > best_f1:
            best_f1, best_state = f1, {n: p.detach().clone() for n, p in model.state_dict().items()}
    model.load_state_dict(best_state)

    # calibrate on validation, report on test
    lv, yv = get_logits(model, dl_va); lt, yt = get_logits(model, dl_te)
    T = fit_temperature(lv, yv)
    p_raw, p_cal = lt.softmax(1), (lt / T).softmax(1)

    os.makedirs(cfg.out, exist_ok=True)
    torch.save(dict(state=best_state, T=T, k=k, acts=ACTS), f"{cfg.out}/nlu_k{k}_s{seed}.pt")
    np.save(f"{cfg.out}/test_probs_k{k}_s{seed}.npy", p_cal.numpy())

    per_class = f1_score(yt, lt.argmax(1), average=None, labels=range(len(ACTS)))
    return dict(k=k, seed=seed, T=T, macro_f1=f1_score(yt, lt.argmax(1), average="macro"),
                ece_raw=ece(p_raw, yt), ece_cal=ece(p_cal, yt), **dict(zip(ACTS, per_class)))

In [14]:
# Set EPOCHS to what the pilot suggested, and use the same number for every k
EPOCHS = 6
cfg.epochs, cfg.out = EPOCHS, "nlu_out"

results = []
for k in [0, 1, 2, 3]:
    for seed in [0, 1, 2]:
        results.append(run(k, seed, tr, va, te))
        pd.DataFrame(results).to_csv(f"{cfg.out}/results_per_run.csv", index=False)  # saves after every run

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 28755.68it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


k=0 seed=0 ep=1 val macro-F1=0.8018
k=0 seed=0 ep=2 val macro-F1=0.8348
k=0 seed=0 ep=3 val macro-F1=0.8410
k=0 seed=0 ep=4 val macro-F1=0.8542
k=0 seed=0 ep=5 val macro-F1=0.8489
k=0 seed=0 ep=6 val macro-F1=0.8594


/tmp/ipykernel_3544681/3455691641.py:16: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  return float(logT.exp())
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 21529.12it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architect

k=0 seed=1 ep=1 val macro-F1=0.8019
k=0 seed=1 ep=2 val macro-F1=0.8308
k=0 seed=1 ep=3 val macro-F1=0.8525
k=0 seed=1 ep=4 val macro-F1=0.8523
k=0 seed=1 ep=5 val macro-F1=0.8677
k=0 seed=1 ep=6 val macro-F1=0.8642


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 20706.48it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


k=0 seed=2 ep=1 val macro-F1=0.7881
k=0 seed=2 ep=2 val macro-F1=0.8374
k=0 seed=2 ep=3 val macro-F1=0.8637
k=0 seed=2 ep=4 val macro-F1=0.8522
k=0 seed=2 ep=5 val macro-F1=0.8577
k=0 seed=2 ep=6 val macro-F1=0.8555


... (14 outputs left out)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 20971.52it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


k=3 seed=1 ep=1 val macro-F1=0.8257
k=3 seed=1 ep=2 val macro-F1=0.8698
k=3 seed=1 ep=3 val macro-F1=0.8996
k=3 seed=1 ep=4 val macro-F1=0.8933
k=3 seed=1 ep=5 val macro-F1=0.9074
k=3 seed=1 ep=6 val macro-F1=0.9142


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 29533.19it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


k=3 seed=2 ep=1 val macro-F1=0.8232
k=3 seed=2 ep=2 val macro-F1=0.8695
k=3 seed=2 ep=3 val macro-F1=0.8917
k=3 seed=2 ep=4 val macro-F1=0.8938
k=3 seed=2 ep=5 val macro-F1=0.9076
k=3 seed=2 ep=6 val macro-F1=0.9061


In [18]:
df = pd.DataFrame(results)
summ = df.groupby("k")[["macro_f1", "ece_raw", "ece_cal"] + ACTS].agg(["mean", "std"]).round(4)
summ.to_csv(f"{cfg.out}/results_summary.csv")
summ[["macro_f1", "ece_raw", "ece_cal"]]

macro_f1         ece_raw         ece_cal        
      mean     std    mean     std    mean     std
k                                                 
0   0.8491  0.0040  0.0600  0.0175  0.0280  0.0022
1   0.8855  0.0049  0.0560  0.0058  0.0274  0.0064
2   0.8914  0.0022  0.0613  0.0013  0.0296  0.0012
3   0.8926  0.0064  0.0581  0.0020  0.0264  0.0024

In [21]:
# Diagnostics for the chosen model: bin sizes, gate trade-off, and accept errors
K, S = 2, 0                                   # same k and seed as Cell 15
P = np.load(f"{cfg.out}/test_probs_k{K}_s{S}.npy")
y = np.array([A2I[r["label"]] for r in te])
conf, pred = P.max(1), P.argmax(1)

# 1) Reliability bins: how many examples sit in each confidence range
print("Confidence bins")
edges = np.linspace(0, 1, 11)
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.sum():
        print(f"  {lo:.1f}-{hi:.1f}  n={m.sum():4d}  conf={conf[m].mean():.3f}  acc={(pred[m]==y[m]).mean():.3f}")

# 2) Gate trade-off: if we only trust predictions above tau, how much do we keep and how accurate are they
print("\nCoverage vs accuracy")
for tau in [0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = conf >= tau
    print(f"  tau={tau}  coverage={keep.mean():.3f}  accuracy on kept={(pred[keep]==y[keep]).mean():.3f}")

# 3) Accept errors: true accepts we missed, and other turns wrongly called accept
print("\nAccept errors (first 30)")
acc_i = A2I["accept"]
errs = [(r, ACTS[p]) for r, p, t in zip(te, pred, y) if (t == acc_i) != (p == acc_i)]
print(f"  total: {len(errs)}")
for r, p in errs[:30]:
    prev = r["hist"][-1][1] if r["hist"] else ""
    print(f"  true={r['label']:8s} pred={p:8s} | prev: {prev[:70]} | {r['text'][:70]}")

Confidence bins
  0.3-0.4  n=   3  conf=0.342  acc=0.667
  0.4-0.5  n=  12  conf=0.484  acc=0.667
  0.5-0.6  n=  36  conf=0.549  acc=0.389
  0.6-0.7  n=  36  conf=0.651  acc=0.472
  0.7-0.8  n=  54  conf=0.752  acc=0.611
  0.8-0.9  n=  94  conf=0.859  acc=0.723
  0.9-1.0  n=1297  conf=0.968  acc=0.959

Coverage vs accuracy
  tau=0.5  coverage=0.990  accuracy on kept=0.907
  tau=0.6  coverage=0.967  accuracy on kept=0.920
  tau=0.7  coverage=0.943  accuracy on kept=0.931
  tau=0.8  coverage=0.908  accuracy on kept=0.943
  tau=0.9  coverage=0.847  accuracy on kept=0.959

Accept errors (first 30)
  total: 64
  true=counter  pred=accept   | prev: How about <PRICE>? I need it for my son. He just got a new job and he  | I can accept it for <PRICE> 
  true=accept   pred=counter  | prev: Oh so nice of you. Can u give me  at <PRICE> | I can do <PRICE> but only if you can pay in all quarters. Meet me at t
  true=accept   pred=counter  | prev: That's pretty steep. I'd be willing to pay more like 

In [20]:
results = pd.read_csv(f"{cfg.out}/results_per_run.csv").to_dict("records")

In [22]:
# Final checks: (1) does history help on rows the relabelling did not touch,
# (2) accept F1 per k, (3) how confident were the accept mistakes
y = np.array([A2I[r["label"]] for r in te])
keep = np.array([r["orig"] == r["label"] for r in te])

print("Macro-F1 on rows the relabelling did not touch")
for k in range(4):
    f = [f1_score(y[keep], np.load(f"{cfg.out}/test_probs_k{k}_s{s}.npy")[keep].argmax(1), average="macro")
         for s in range(3)]
    print(f"  k={k}  {np.mean(f):.4f} ± {np.std(f):.4f}")

print("\nAccept F1 by k (test split)")
print(df.groupby("k")["accept"].agg(["mean", "std"]).round(4))

print("\nConfidence of accept mistakes (k=2, seed 0)")
P = np.load(f"{cfg.out}/test_probs_k2_s0.npy")
conf, pred = P.max(1), P.argmax(1)
acc_i = A2I["accept"]
wrong = (y == acc_i) != (pred == acc_i)
print(f"  mistakes: {wrong.sum()}  conf >= 0.9: {(conf[wrong] >= 0.9).sum()}  conf < 0.8: {(conf[wrong] < 0.8).sum()}")

Macro-F1 on rows the relabelling did not touch
  k=0  0.8419 ± 0.0039
  k=1  0.8706 ± 0.0038
  k=2  0.8792 ± 0.0040
  k=3  0.8810 ± 0.0065

Accept F1 by k (test split)
     mean     std
k                
0  0.6973  0.0269
1  0.7474  0.0255
2  0.7521  0.0232
3  0.7337  0.0234

Confidence of accept mistakes (k=2, seed 0)
  mistakes: 64  conf >= 0.9: 24  conf < 0.8: 25


In [23]:
_cache = {}
def load_nlu(path):
    # load each checkpoint once and reuse it
    if path not in _cache:
        ck = torch.load(path, map_location=dev)
        m = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=len(ACTS))
        m.load_state_dict(ck["state"]); m.to(dev).eval()
        _cache[path] = (m, ck["T"], ck["k"])
    return _cache[path]

@torch.no_grad()
def predict(history, message, path="nlu_out/nlu_k2_s0.pt"):
    """history: list of (speaker, text) with speaker 'BUYER' or 'SELLER'.
    Returns {act: calibrated probability}."""
    m, T, k = load_nlu(path)
    ctx = " ".join(f"[{w}] {mask_prices(t)}" for w, t in history[-k:]) if k > 0 else None
    enc = tok("[SELLER] " + mask_prices(message), ctx, truncation="longest_first",
              max_length=cfg.max_len, return_tensors="pt").to(dev)
    p = (m(**enc).logits / T).softmax(-1).squeeze(0).cpu().numpy()
    return dict(zip(ACTS, p.round(4).tolist()))

print(predict([("BUYER", "Can you do $40?"), ("SELLER", "I can do 45")], "ok that works"))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 25085.55it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'propose': 0.00139999995008111, 'counter': 0.006200000178068876, 'accept': 0.9785000085830688, 'reject': 0.003700000001117587, 'inquire': 0.0019000000320374966, 'vague': 0.008200000040233135}


In [24]:
# Quick sanity test: each case should lean towards the act in the comment
h = [("BUYER", "Can you do $40?"), ("SELLER", "I can do 45")]
tests = [
    (h, "ok that works"),                        # accept
    (h, "no way, that's too much"),              # reject
    (h, "how about 42?"),                        # counter
    (h, "does it come with the charger?"),       # inquire
    ([], "ok that works"),                       # no offer to accept -> should NOT be a confident accept
    ([], "I'm selling it for $50"),              # propose
]
for hist, msg in tests:
    p = predict(hist, msg)
    top = max(p, key=p.get)
    print(f"{top:8s} {p[top]:.2f} | history={'yes' if hist else 'none'} | {msg}")

accept   0.98 | history=yes | ok that works
reject   0.70 | history=yes | no way, that's too much
counter  0.94 | history=yes | how about 42?
inquire  0.98 | history=yes | does it come with the charger?
vague    0.71 | history=none | ok that works
propose  0.97 | history=none | I'm selling it for $50
